# A Self-Interpreter for SKI, in SKI — with the Aviary kernel

Same material as `ski-in-ski.ipynb`, but every cell is combinatory logic
rather than Python. Install and register the kernel:

    pip install aviary-kernel==0.1.0
    python -m aviary_kernel.install

Syntax: `NAME VAR... -> body` defines a rule, `NAME := expr` an alias, a
bare expression reduces, `%magic` for kernel commands, `#` for comments.

**Why a self-interpreter here is not routine.** SKI cannot inspect its
own terms. Jay & Given-Wilson (JSL 76(3), 2011, Thm 3.2) proved that
recovering `X` from `S K X` is not representable by any SK-combinator: a
calculus of pure functions can *apply* an argument, never look at one.
We buy the missing capability in the encoding rather than the calculus.

In [ ]:
# S K K and I are the same function...
S K K x
I x

...but different terms. `%whatis` shows a definition three ways, and
`%ski` expands anything to the pure basis.

In [ ]:
%whatis B
%ski B

## 1. Scott encoding

Encode a term as something that dispatches on *itself*: one continuation
per constructor, and the matching one is applied.

    <S>   = λ s k i z. s
    <K>   = λ s k i z. k
    <I>   = λ s k i z. i
    <t u> = λ s k i z. z <t> <u>

In [ ]:
encS s k i z -> s
encK s k i z -> k
encI s k i z -> i
encA t u s k i z -> z t u

In [ ]:
# each constructor selects its own continuation
encS c1 c2 c3 c4
encK c1 c2 c3 c4
encI c1 c2 c3 c4
encA encS encK c1 c2 c3 c4

Case analysis in one application. Note the cost, though — `%ski` shows
what the application constructor really is:

In [ ]:
%ski encA

## 2. Scott data, and the unquoter

`UQ` interprets each constructor as *itself* and application as
application, releasing the encoded term back into the host. It is the
left inverse of encoding — a **self-recogniser** — and doubles as our
decoder.

In [ ]:
nil n c -> n
cons h t n c -> c h t
nothing n j -> n
just x n j -> j x
zero z s -> z
suc n z s -> s n

In [ ]:
UQA f t u -> (f t) (f u)
UQGen f m -> m S K I (UQA f)
UQ := Y UQGen

In [ ]:
UQ (encA encS encK)
UQ (encA (encA encS encK) encK)

## 3. Taking a term apart

`sp` unwinds the spine collecting arguments into a list; `rb` rebuilds an
application from a head and a list.

In [ ]:
resS acc s k i -> s acc
resK acc s k i -> k acc
resI acc s k i -> i acc
spApp f acc t u -> f t (cons u acc)
spGen f m acc -> m (resS acc) (resK acc) (resI acc) (spApp f acc)
sp := Y spGen
rb1 f h x xs -> f (encA h x) xs
rbGen f h args -> args h (rb1 f h)
rb := Y rbGen

## 4. One step

`stp` performs a single normal-order contraction: `just <t'>` if a rule
fired, `nothing` if the head is under-applied. Four ordinary case
analyses on a data structure — this is what the encoding bought.

In [ ]:
stepI1 x rest -> just (rb x rest)
stepI args -> args nothing stepI1
stepK2 x y rest -> just (rb x rest)
stepK1 x r -> r nothing (stepK2 x)
stepK args -> args nothing stepK1
stepS3 x y z rest -> just (rb (encA (encA x z) (encA y z)) rest)
stepS2 x y r2 -> r2 nothing (stepS3 x y)
stepS1 x r -> r nothing (stepS2 x)
stepS args -> args nothing stepS1
stp m -> sp m nil stepS stepK stepI

In [ ]:
# apply the Maybe to two continuations to read it
stp (encA encI encK) NO JU
stp (encA (encA encK encS) encI) NO JU

## 5. The fueled loop

`whnfF n <t>` runs up to `n` object steps. Fuel makes the interpreter
total and introduces a third outcome: the object program can now *time
out* as well as return. Reading a result is one application — `TIMEOUT`
for `nothing`, `UQ` for `just`.

In [ ]:
wf1 w m n2 -> stp m (just m) (w n2)
wfGen w n m -> n nothing (wf1 w m)
whnfF := Y wfGen

In [ ]:
n1 := suc zero
n2 := suc n1
n3 := suc n2
n5 := suc (suc n3)
n6 := suc n5
whnfF n3 (encA encI encK) TIMEOUT UQ
whnfF n5 (encA (encA (encA encS encK) encK) encI) TIMEOUT UQ

## 6. Divergence, and what fuel can and cannot tell you

`Ω = (S I I)(S I I)` has no normal form. But watch: `I⁵K` is
indistinguishable from it at fuel 5, and converges at 6.

**A finite run cannot establish divergence, only refute it.** That is why
the outcome set is `{value, timeout}` rather than `{value, divergence}` —
timeout is an observation, divergence is a claim about all
observations.

In [ ]:
qSII := encA (encA encS encI) encI
qOM := encA qSII qSII
qI5K := encA encI (encA encI (encA encI (encA encI (encA encI encK))))
whnfF n5 qOM TIMEOUT UQ
whnfF n5 qI5K TIMEOUT UQ
whnfF n6 qI5K TIMEOUT UQ

## 7. Object-level laziness survives

`K I Ω` discards `Ω` unevaluated. The emulator inherits this, because the
object-level `K` rule discards the argument before the interpreter ever
recurses into it.

In [ ]:
n10 := suc (suc (suc (suc (suc n5))))
n20 := suc (suc (suc (suc (suc (suc (suc (suc (suc (suc n10)))))))))
qKIOM := encA (encA encK encI) qOM
whnfF n20 qKIOM TIMEOUT UQ

## 8. The wall

The Python notebook goes one level further: it encodes `whnfF` itself and
runs the interpreter on the interpreter, 504,930 contractions.

**That cell cannot be written here, and the reason is the whole point.**
To quote `whnfF` you need a function from terms to their encodings — and
quoting must separate terms that are extensionally equal (`S K K` and `I`
denote the same function but must encode differently). Jay &
Given-Wilson's Theorem 3.2 says no SK-combinator does that. In the Python
notebook `enc` is *Python*; that is not a convenience, it is the one step
that provably must come from outside the calculus.

So the notebook you are reading can build a self-interpreter and run it,
but cannot hand it its own text. Everything above the encoder is
internal — including `UQ`, the decoder, 43 atoms of ordinary SKI. Only
the encoder is meta.

Two practical notes for anyone extending this. Terms built with `Y`
should be reduced with `%whnf`; full normalisation unrolls the fixpoint
forever in argument position. And the `Err`-constructor material from the
Python notebook (an added operator whose meaning the interpreter must
author — four atoms to expose it, seven to hide it) is left there rather
than reproduced here.

### References

- C. B. Jay and T. Given-Wilson, *A combinatory account of internal
  structure*, JSL 76(3):807–826, 2011.
- C. B. Jay and J. Palsberg, *Typed self-interpretation by pattern
  matching*, ICFP 2011, 247–258.
- T. Æ. Mogensen, *Efficient self-interpretations in lambda calculus*,
  JFP 2(3):345–363, 1992.